In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# ============================================
# BYTE PAIR ENCODING (BPE) TOKENIZER
# nanoGPT-style minimal tokenizer
# ============================================

from collections import defaultdict


# ============================================
# Count frequency of adjacent token pairs
# ============================================

def get_stats(ids):
    """
    Given a list of token ids, return counts of consecutive pairs.

    Example:
    ids = [1, 2, 3, 1, 2]

    pairs:
    (1,2), (2,3), (3,1), (1,2)

    returns:
    {
        (1,2): 2,
        (2,3): 1,
        (3,1): 1
    }
    """

    counts = defaultdict(int)

    # zip(ids, ids[1:])
    # creates consecutive pairs
    #
    # Example:
    # ids         = [1,2,3,4]
    # ids[1:]     = [2,3,4]
    #
    # zip(...) => [(1,2), (2,3), (3,4)]

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1

    return counts


# ============================================
# Merge all occurrences of a pair
# ============================================

def merge(ids, pair, idx):
    """
    Replace every occurrence of `pair`
    in ids with new token idx.

    Example:
    ids  = [1,2,3,1,2]
    pair = (1,2)
    idx  = 256

    result:
    [256,3,256]
    """

    new_ids = []

    i = 0

    while i < len(ids):

        # Check if current pair matches
        if (
            i < len(ids) - 1 and
            ids[i] == pair[0] and
            ids[i + 1] == pair[1]
        ):

            # Replace pair with new token
            new_ids.append(idx)

            # Skip next token because merged
            i += 2

        else:
            # Keep token as-is
            new_ids.append(ids[i])
            i += 1

    return new_ids


# ============================================
# Basic BPE Tokenizer
# ============================================

class BasicTokenizer:

    def __init__(self):

        # ------------------------------------
        # merges:
        # maps pair -> new token id
        #
        # Example:
        # (104,105) -> 256
        # meaning:
        # b'h' + b'i' -> token 256
        # ------------------------------------

        self.merges = {}

        # ------------------------------------
        # vocab:
        # maps token id -> bytes
        #
        # Initially:
        # 0-255 are raw bytes
        # ------------------------------------

        self.vocab = {
            i: bytes([i]) for i in range(256)
        }

    # ========================================
    # TRAIN
    # ========================================

    def train(self, text, vocab_size, verbose=False):

        """
        Learn merge rules from text.

        vocab_size:
            total desired vocab size

        Since initial vocab already has
        256 byte tokens:

            num_merges = vocab_size - 256
        """

        assert vocab_size >= 256

        num_merges = vocab_size - 256

        # ------------------------------------
        # Convert text -> raw UTF-8 bytes
        # ------------------------------------

        ids = list(text.encode("utf-8"))

        # ------------------------------------
        # Perform merges iteratively
        # ------------------------------------

        for i in range(num_merges):

            # Count pair frequencies
            stats = get_stats(ids)

            # No pairs left
            if len(stats) == 0:
                break

            # Most frequent pair
            pair = max(stats, key=stats.get)

            # New token id
            idx = 256 + i

            # Replace occurrences
            ids = merge(ids, pair, idx)

            # Store merge rule
            self.merges[pair] = idx

            # Build new vocab entry
            #
            # Example:
            # vocab[256] = b'h' + b'i'
            #
            self.vocab[idx] = (
                self.vocab[pair[0]] +
                self.vocab[pair[1]]
            )

            if verbose:
                print(
                    f"merge {i+1}/{num_merges}: "
                    f"{pair} -> {idx} "
                    f"({self.vocab[idx]}) "
                    f"had {stats[pair]} occurrences"
                )

    # ========================================
    # ENCODE
    # ========================================

    def encode(self, text):

        """
        Convert text -> token ids
        using learned merges.

        IMPORTANT:
        merges must be applied
        in TRAINING ORDER.
        """

        # Raw bytes
        ids = list(text.encode("utf-8"))

        while len(ids) >= 2:

            # Current pair statistics
            stats = get_stats(ids)

            # --------------------------------
            # Find pair with LOWEST merge idx
            #
            # This ensures:
            # earliest learned merge applied first
            # --------------------------------

            pair = min(
                stats,
                key=lambda p: self.merges.get(
                    p,
                    float("inf")
                )
            )

            # If pair was never learned
            if pair not in self.merges:
                break

            # Apply merge
            idx = self.merges[pair]

            ids = merge(ids, pair, idx)

        return ids

    # ========================================
    # DECODE
    # ========================================

    def decode(self, ids):

        """
        Convert token ids -> text
        """

        # Recover raw bytes
        tokens = b"".join(
            self.vocab[i] for i in ids
        )

        # UTF-8 decode safely
        #
        # errors='replace'
        # avoids crashes on malformed UTF-8
        #
        return tokens.decode(
            "utf-8",
            errors="replace"
        )


# ============================================
# TESTS
# ============================================

def test_ascii_invariant(tokenizer):

    """
    Proof-of-correctness test:

    decode(encode(x)) == x

    for all ASCII characters
    """

    for i in range(128):

        s = chr(i)

        encoded = tokenizer.encode(s)

        decoded = tokenizer.decode(encoded)

        assert decoded == s, (
            f"FAILED for ASCII {i}\n"
            f"original : {repr(s)}\n"
            f"decoded  : {repr(decoded)}"
        )

    print("ASCII invariant test PASSED")


def test_string(tokenizer, text):

    encoded = tokenizer.encode(text)

    decoded = tokenizer.decode(encoded)

    print("=" * 50)
    print("TEXT:")
    print(repr(text))

    print("\nENCODED IDS:")
    print(encoded)

    print("\nDECODED:")
    print(repr(decoded))

    print("\nMATCH:")
    print(decoded == text)

    assert decoded == text


# ============================================
# EXAMPLE USAGE
# ============================================

if __name__ == "__main__":

    text = """
    hello world.
    this is a tiny bpe tokenizer.
    hello hello hello.
    nanogpt is fun.
    """

    # Create tokenizer
    tokenizer = BasicTokenizer()

    # Train tokenizer
    tokenizer.train(
        text=text,
        vocab_size=300,
        verbose=True
    )

    # ----------------------------------------
    # Test correctness invariant
    # ----------------------------------------

    test_ascii_invariant(tokenizer)

    # ----------------------------------------
    # Test sample strings
    # ----------------------------------------

    test_string(
        tokenizer,
        "hello world"
    )

    test_string(
        tokenizer,
        "nanogpt"
    )

    test_string(
        tokenizer,
        "😀 unicode test 😀"
    )

    # ----------------------------------------
    # Manual encode/decode
    # ----------------------------------------

    s = "transformers are awesome"

    ids = tokenizer.encode(s)

    recovered = tokenizer.decode(ids)

    print("\n" + "=" * 50)
    print("Manual Example")
    print("=" * 50)

    print("Original:")
    print(s)

    print("\nToken IDs:")
    print(ids)

    print("\nRecovered:")
    print(recovered)

merge 1/44: (32, 32) -> 256 (b'  ') had 15 occurrences
merge 2/44: (10, 256) -> 257 (b'\n  ') had 5 occurrences
merge 3/44: (257, 256) -> 258 (b'\n    ') had 5 occurrences
merge 4/44: (104, 101) -> 259 (b'he') had 4 occurrences
merge 5/44: (259, 108) -> 260 (b'hel') had 4 occurrences
merge 6/44: (260, 108) -> 261 (b'hell') had 4 occurrences
merge 7/44: (261, 111) -> 262 (b'hello') had 4 occurrences
merge 8/44: (46, 258) -> 263 (b'.\n    ') had 4 occurrences
merge 9/44: (262, 32) -> 264 (b'hello ') had 3 occurrences
merge 10/44: (105, 115) -> 265 (b'is') had 3 occurrences
merge 11/44: (265, 32) -> 266 (b'is ') had 3 occurrences
merge 12/44: (32, 116) -> 267 (b' t') had 2 occurrences
merge 13/44: (258, 264) -> 268 (b'\n    hello ') had 1 occurrences
merge 14/44: (268, 119) -> 269 (b'\n    hello w') had 1 occurrences
merge 15/44: (269, 111) -> 270 (b'\n    hello wo') had 1 occurrences
merge 16/44: (270, 114) -> 271 (b'\n    hello wor') had 1 occurrences
merge 17/44: (271, 108) -> 272 (b'\

In [6]:
# ============================================
# GPT-2 STYLE BPE TOKENIZER
# WITH REGEX PRETOKENIZATION
# ============================================

import regex as re
from collections import defaultdict


# ============================================
# GPT-2 regex pattern
# ============================================

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"""


# ============================================
# Count adjacent token pairs
# ============================================

def get_stats(ids):

    counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1

    return counts


# ============================================
# Merge token pairs
# ============================================

def merge(ids, pair, idx):

    new_ids = []

    i = 0

    while i < len(ids):

        if (
            i < len(ids) - 1 and
            ids[i] == pair[0] and
            ids[i + 1] == pair[1]
        ):

            new_ids.append(idx)
            i += 2

        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


# ============================================
# GPT2 BPE Tokenizer
# ============================================

class BasicTokenizer:

    def __init__(self):

        # ------------------------------------
        # merge rules:
        # (token_a, token_b) -> new_token
        # ------------------------------------

        self.merges = {}

        # ------------------------------------
        # vocab:
        # token_id -> bytes
        # ------------------------------------

        self.vocab = {
            i: bytes([i]) for i in range(256)
        }

        # compiled regex
        self.compiled_pattern = re.compile(GPT2_SPLIT_PATTERN)

    # ========================================
    # TRAIN
    # ========================================

    def train(self, text, vocab_size, verbose=False):

        assert vocab_size >= 256

        num_merges = vocab_size - 256

        # ------------------------------------
        # GPT2 PRETOKENIZATION
        # ------------------------------------

        chunks = re.findall(
            self.compiled_pattern,
            text
        )

        # ------------------------------------
        # Convert chunks -> byte ids
        # ------------------------------------

        ids = []

        for chunk in chunks:
            chunk_bytes = list(chunk.encode("utf-8"))
            ids.extend(chunk_bytes)

        # ------------------------------------
        # Learn merges
        # ------------------------------------

        for i in range(num_merges):

            stats = get_stats(ids)

            if len(stats) == 0:
                break

            pair = max(stats, key=stats.get)

            idx = 256 + i

            ids = merge(ids, pair, idx)

            self.merges[pair] = idx

            self.vocab[idx] = (
                self.vocab[pair[0]] +
                self.vocab[pair[1]]
            )

            if verbose:
                print(
                    f"merge {i+1}/{num_merges}: "
                    f"{pair} -> {idx} "
                    f"{self.vocab[idx]}"
                )

    # ========================================
    # ENCODE SINGLE CHUNK
    # ========================================

    def _encode_chunk(self, chunk_bytes):

        ids = list(chunk_bytes)

        while len(ids) >= 2:

            stats = get_stats(ids)

            # earliest learned merge
            pair = min(
                stats,
                key=lambda p: self.merges.get(
                    p,
                    float("inf")
                )
            )

            if pair not in self.merges:
                break

            idx = self.merges[pair]

            ids = merge(ids, pair, idx)

        return ids

    # ========================================
    # GPT2 ENCODE
    # ========================================

    def encode(self, text):

        # ------------------------------------
        # Split into GPT2 regex chunks
        # ------------------------------------

        chunks = re.findall(
            self.compiled_pattern,
            text
        )

        ids = []

        # ------------------------------------
        # Encode each chunk independently
        # ------------------------------------

        for chunk in chunks:

            chunk_bytes = chunk.encode("utf-8")

            chunk_ids = self._encode_chunk(
                chunk_bytes
            )

            ids.extend(chunk_ids)

        return ids

    # ========================================
    # DECODE
    # ========================================

    def decode(self, ids):

        tokens = b"".join(
            self.vocab[i] for i in ids
        )

        return tokens.decode(
            "utf-8",
            errors="replace"
        )


# ============================================
# TESTS
# ============================================

def test_ascii_invariant(tokenizer):

    for i in range(128):

        s = chr(i)

        encoded = tokenizer.encode(s)

        decoded = tokenizer.decode(encoded)

        assert decoded == s, (
            f"FAILED ASCII {i}"
        )

    print("ASCII invariant PASSED")


def test_roundtrip(tokenizer, text):

    encoded = tokenizer.encode(text)

    decoded = tokenizer.decode(encoded)

    print("=" * 60)

    print("TEXT:")
    print(repr(text))

    print("\nENCODED:")
    print(encoded)

    print("\nDECODED:")
    print(repr(decoded))

    print("\nMATCH:")
    print(decoded == text)

    assert decoded == text


# ============================================
# MAIN
# ============================================

if __name__ == "__main__":

    text = """
    Hello world!
    GPT-2 style tokenization is awesome.
    dog dog dog
    bulldog bulldog
    😀 emojis work too 😀
    """

    tokenizer = BasicTokenizer()

    tokenizer.train(
        text=text,
        vocab_size=300,
        verbose=True
    )

    # ----------------------------------------
    # invariant tests
    # ----------------------------------------

    test_ascii_invariant(tokenizer)

    # ----------------------------------------
    # roundtrip tests
    # ----------------------------------------

    test_roundtrip(
        tokenizer,
        "Hello GPT-2!"
    )

    test_roundtrip(
        tokenizer,
        " dog"
    )

    test_roundtrip(
        tokenizer,
        "bulldog"
    )

    test_roundtrip(
        tokenizer,
        "😀 unicode 😀"
    )

merge 1/44: (32, 32) -> 256 b'  '
merge 2/44: (10, 256) -> 257 b'\n  '
merge 3/44: (257, 256) -> 258 b'\n    '
merge 4/44: (100, 111) -> 259 b'do'
merge 5/44: (259, 103) -> 260 b'dog'
merge 6/44: (108, 108) -> 261 b'll'
merge 7/44: (260, 32) -> 262 b'dog '
merge 8/44: (111, 32) -> 263 b'o '
merge 9/44: (119, 111) -> 264 b'wo'
merge 10/44: (264, 114) -> 265 b'wor'
merge 11/44: (32, 116) -> 266 b' t'
merge 12/44: (266, 111) -> 267 b' to'
merge 13/44: (105, 115) -> 268 b'is'
merge 14/44: (268, 32) -> 269 b'is '
merge 15/44: (260, 258) -> 270 b'dog\n    '
merge 16/44: (98, 117) -> 271 b'bu'
merge 17/44: (271, 261) -> 272 b'bull'
merge 18/44: (240, 159) -> 273 b'\xf0\x9f'
merge 19/44: (273, 152) -> 274 b'\xf0\x9f\x98'
merge 20/44: (274, 128) -> 275 b'\xf0\x9f\x98\x80'
merge 21/44: (258, 72) -> 276 b'\n    H'
merge 22/44: (276, 101) -> 277 b'\n    He'
merge 23/44: (277, 261) -> 278 b'\n    Hell'
merge 24/44: (278, 263) -> 279 b'\n    Hello '
merge 25/44: (279, 265) -> 280 b'\n    Hello wor'


In [7]:
import json
import regex as re
from functools import lru_cache


# =========================================================
# GPT2 regex
# =========================================================

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+"""


# =========================================================
# BYTE <-> UNICODE MAPPING
# =========================================================

@lru_cache()
def bytes_to_unicode():
    """
    GPT2 reversible byte->unicode mapping.
    """

    bs = (
        list(range(ord("!"), ord("~") + 1)) +
        list(range(ord("¡"), ord("¬") + 1)) +
        list(range(ord("®"), ord("ÿ") + 1))
    )

    cs = bs[:]

    n = 0

    for b in range(256):

        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1

    cs = [chr(n) for n in cs]

    return dict(zip(bs, cs))


# =========================================================
# GPT2 TOKENIZER
# =========================================================

class GPT2Tokenizer:

    def __init__(self, encoder_path, bpe_path):

        # ---------------------------------------------
        # byte encoder / decoder
        # ---------------------------------------------

        self.byte_encoder = bytes_to_unicode()

        self.byte_decoder = {
            v: k for k, v in self.byte_encoder.items()
        }

        # ---------------------------------------------
        # load vocab
        # ---------------------------------------------

        with open(encoder_path, "r", encoding="utf-8") as f:
            self.encoder = json.load(f)

        self.decoder = {
            v: k for k, v in self.encoder.items()
        }

        # ---------------------------------------------
        # load merge rules
        # ---------------------------------------------

        with open(bpe_path, "r", encoding="utf-8") as f:

            merges = f.read().split("\n")[1:-1]

        merges = [
            tuple(merge.split())
            for merge in merges
        ]

        # rank = training order
        self.bpe_ranks = dict(zip(
            merges,
            range(len(merges))
        ))

        # regex
        self.pattern = re.compile(GPT2_SPLIT_PATTERN)

    # =====================================================
    # helper: get adjacent symbol pairs
    # =====================================================

    def get_pairs(self, word):

        pairs = set()

        prev_char = word[0]

        for char in word[1:]:

            pairs.add((prev_char, char))

            prev_char = char

        return pairs

    # =====================================================
    # BPE
    # =====================================================

    @lru_cache(maxsize=None)
    def bpe(self, token):

        """
        Apply GPT2 BPE merges to one token.
        """

        word = tuple(token)

        pairs = self.get_pairs(word)

        if not pairs:
            return token

        while True:

            # choose earliest merge
            bigram = min(
                pairs,
                key=lambda pair: self.bpe_ranks.get(
                    pair,
                    float("inf")
                )
            )

            if bigram not in self.bpe_ranks:
                break

            first, second = bigram

            new_word = []

            i = 0

            while i < len(word):

                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j

                except:
                    new_word.extend(word[i:])
                    break

                if (
                    word[i] == first and
                    i < len(word) - 1 and
                    word[i + 1] == second
                ):

                    new_word.append(first + second)

                    i += 2

                else:
                    new_word.append(word[i])
                    i += 1

            word = tuple(new_word)

            if len(word) == 1:
                break

            pairs = self.get_pairs(word)

        return " ".join(word)

    # =====================================================
    # ENCODE
    # =====================================================

    def encode(self, text):

        bpe_tokens = []

        # regex chunks
        for token in re.findall(self.pattern, text):

            # bytes -> unicode chars
            token = "".join(
                self.byte_encoder[b]
                for b in token.encode("utf-8")
            )

            # apply BPE
            bpe_result = self.bpe(token)

            # convert to token ids
            bpe_tokens.extend(
                self.encoder[bpe_token]
                for bpe_token in bpe_result.split(" ")
            )

        return bpe_tokens

    # =====================================================
    # DECODE
    # =====================================================

    def decode(self, tokens):

        text = "".join(
            self.decoder[token]
            for token in tokens
        )

        # unicode chars -> bytes
        byte_array = bytearray([
            self.byte_decoder[c]
            for c in text
        ])

        return byte_array.decode(
            "utf-8",
            errors="replace"
        )

In [8]:
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json
!wget https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe

--2026-04-28 18:11:57--  https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.60.179.33
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.60.179.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1042301 (1018K) [application/json]
Saving to: ‘encoder.json’

encoder.json        100%[===================>]   1018K  4.76MB/s    in 0.2s    

2026-04-28 18:11:58 (4.76 MB/s) - ‘encoder.json’ saved [1042301/1042301]

--2026-04-28 18:11:58--  https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe
Resolving openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)... 20.60.179.33
Connecting to openaipublic.blob.core.windows.net (openaipublic.blob.core.windows.net)|20.60.179.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 456318 (446K) [application/octet-stream]
Savin

In [9]:
tokenizer = GPT2Tokenizer(
    "encoder.json",
    "vocab.bpe"
)

In [10]:
text = "Hello world!"

ids = tokenizer.encode(text)

print(ids)

decoded = tokenizer.decode(ids)

print(decoded)

[15496, 995, 0]
Hello world!


In [11]:
!pip install tiktoken

In [12]:
import tiktoken
import pandas as pd

enc = tiktoken.get_encoding("gpt2")

test_strings = [
    "Hello world!",
    "GPT-2 is awesome.",
    "1234567890",
    "def hello(): return 42",
    "😀 emojis 😀",
    " spaces matter",
    "dog",
    " dog",
    "a+b=c",
    "What's going on?!"
]

rows = []

for s in test_strings:

    mine = tokenizer.encode(s)

    theirs = enc.encode(s)

    rows.append({
        "text": s,
        "my_tokens": mine,
        "tiktoken": theirs,
        "match": mine == theirs
    })

df = pd.DataFrame(rows)

print(df)

                     text                                  my_tokens  \
0            Hello world!                            [15496, 995, 0]   
1       GPT-2 is awesome.         [38, 11571, 12, 17, 318, 7427, 13]   
2              1234567890                  [10163, 29228, 40401, 15]   
3  def hello(): return 42      [4299, 23748, 33529, 1441, 220, 3682]   
4              😀 emojis 😀  [47249, 222, 795, 13210, 271, 30325, 222]   
5           spaces matter                               [9029, 2300]   
6                     dog                                     [9703]   
7                     dog                                     [3290]   
8                   a+b=c                       [64, 10, 65, 28, 66]   
9       What's going on?!              [2061, 338, 1016, 319, 12248]   

                                    tiktoken  match  
0                            [15496, 995, 0]   True  
1         [38, 11571, 12, 17, 318, 7427, 13]   True  
2                 [10163, 2231, 30924, 3829] 